In [ ]:
# Define file paths
import pandas as pd 
import os 
behaviors_path = "../data/raw/input/mind-news-dataset/MINDsmall_train/behaviors.tsv"
entity_embedding_path = "../data/raw/input/mind-news-dataset/MINDsmall_train/entity_embedding.vec"
news_path = "../data/raw/input/mind-news-dataset/MINDsmall_train/news.tsv"
relation_embedding_path = "../data/raw/input/mind-news-dataset/MINDsmall_train/relation_embedding.vec"

# Load behaviors data
behaviors = pd.read_csv(behaviors_path, sep='\t', header=None, names=['ImpressionID', 'UserID', 'Time', 'History', 'Impressions'])

# Load news data
news = pd.read_csv(news_path, sep='\t', header=None, names=['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'TitleEntities', 'AbstractEntities'])

# Load entity embeddings
entity_embeddings = {}
with open(entity_embedding_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        entity_embeddings[parts[0]] = list(map(float, parts[1:]))

# Load relation embeddings
relation_embeddings = {}
with open(relation_embedding_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        relation_embeddings[parts[0]] = list(map(float, parts[1:]))

# Preprocess behaviors data
behaviors['History'] = behaviors['History'].fillna('').apply(lambda x: x.split(' '))
behaviors['Impressions'] = behaviors['Impressions'].apply(lambda x: [imp.split('-') for imp in x.split(' ')])

# Create user profiles
user_profiles = {}
for _, row in behaviors.iterrows():
    user_id = row['UserID']
    history = row['History']
    if user_id not in user_profiles:
        user_profiles[user_id] = []
    user_profiles[user_id].extend(history)

print("Data loaded and processed successfully.")

Data loaded and processed successfully.


In [2]:
print("\nNews Dataset:")
news.head()


News Dataset:


,NewsID,Category,SubCategory,Title,Abstract,URL,TitleEntities,AbstractEntities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [3]:
print("\nUser Profiles Summary:")
user_profile_lengths = {user: len(history) for user, history in user_profiles.items()}
print(f"Total Users: {len(user_profiles)}")
print(f"Average Profile Length: {sum(user_profile_lengths.values()) / len(user_profiles):.2f}")


User Profiles Summary:
Total Users: 50000
Average Profile Length: 102.22


# CSV file

NameError: name 'user_profiles' is not defined

# Drop Null 

In [4]:
# Remove rows with null values in the news dataset
news_cleaned = news.dropna()

# Encode categorical variables in the news dataset
news_cleaned['Category'] = news_cleaned['Category'].astype('category').cat.codes
news_cleaned['SubCategory'] = news_cleaned['SubCategory'].astype('category').cat.codes

# Prepare user profiles by removing duplicates and ensuring unique histories
user_profiles_cleaned = {user: list(set(history)) for user, history in user_profiles.items()}

# Display the cleaned datasets
print("Cleaned News Dataset:")
print(news_cleaned.head())

print("\nCleaned User Profiles:")
for user, history in list(user_profiles_cleaned.items())[:5]:  # Display first 5 user profiles
    print(f"User {user}: {history}")

<ipython-input-4-1a091df46598>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news_cleaned['Category'] = news_cleaned['Category'].astype('category').cat.codes
<ipython-input-4-1a091df46598>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news_cleaned['SubCategory'] = news_cleaned['SubCategory'].astype('category').cat.codes


Cleaned News Dataset:
   NewsID  Category  SubCategory  \
0  N55528         6          138   
1  N19639         4          254   
2  N61837        10          186   
3  N53526         4          249   
4  N38324         4          147   

                                               Title  \
0  The Brands Queen Elizabeth, Prince Charles, an...   
1                      50 Worst Habits For Belly Fat   
2  The Cost of Trump's Aid Freeze in the Trenches...   
3  I Was An NBA Wife. Here's How It Affected My M...   
4  How to Get Rid of Skin Tags, According to a De...   

                                            Abstract  \
0  Shop the notebooks, jackets, and more that the...   
1  These seemingly harmless habits are holding yo...   
2  Lt. Ivan Molchanets peeked over a parapet of s...   
3  I felt like I was a fraud, and being an NBA wi...   
4  They seem harmless, but there's a very good re...   

                                             URL  \
0  https://assets.msn.com/labs/mind

# Handel Text

In [5]:
import re

# Function to clean and tokenize sentences using regular expressions
def process_sentences(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Split into words
    words = text.split()
    return words

# Apply the function to the 'Title' column of the news dataset
news['ProcessedTitle'] = news['Title'].apply(lambda x: process_sentences(x))

# Display the processed titles
print("Processed Titles:")
news[['Title', 'ProcessedTitle']].head()

Processed Titles:


,Title,ProcessedTitle
0,"The Brands Queen Elizabeth, Prince Charles, an...","[the, brands, queen, elizabeth, prince, charle..."
1,50 Worst Habits For Belly Fat,"[worst, habits, for, belly, fat]"
2,The Cost of Trump's Aid Freeze in the Trenches...,"[the, cost, of, trumps, aid, freeze, in, the, ..."
3,I Was An NBA Wife. Here's How It Affected My M...,"[i, was, an, nba, wife, heres, how, it, affect..."
4,"How to Get Rid of Skin Tags, According to a De...","[how, to, get, rid, of, skin, tags, according,..."
